# Notebook para treinamento e testes de modelos

In [1]:
import pandas as pd
import numpy as np

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
import prophet

from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import datetime
from dateutil.relativedelta import relativedelta
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.ERROR)

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
horizonte_previsao = 3
tamanho_teste = 6
n_trials = 3
metrica_erro = 'mae'
tolerancia_fipe = 200
tolerancia_exog = 0.01

skus_teste = [0, 1, 2, 3, 4]

## Leitura da base de dados

In [3]:
# Base fipe historica
fipe_path = './data/dados_fipe_tratados.csv'
# Base da taxa de cambio
exchange_path = './data/DEXBZUS_tratados.csv'
# Base IPCA
ipca_path = './data/bcdata.sgs.433_tratados.csv' 

df_fipe = pd.read_csv(fipe_path)
print('FIPE shape:', df_fipe.shape)
print(df_fipe.head())

df_ex = pd.read_csv(exchange_path)
print('DEXBZUS head:')
print(df_ex.head())

df_ipca = pd.read_csv(ipca_path)
print('IPCA head:')
print(df_ipca.head())

FIPE shape: (411498, 9)
   Unnamed: 0       reference_date brand_name          model_name  year  \
0           0  2021-01-01 00:00:00       Fiat           147 C/ CL  1987   
1           1  2021-01-01 00:00:00       Fiat           147 C/ CL  1986   
2           2  2021-01-01 00:00:00       Fiat           147 C/ CL  1985   
3           3  2021-01-01 00:00:00       Fiat  147 Furgão (todos)  1987   
4           4  2021-01-01 00:00:00       Fiat  147 Furgão (todos)  1986   

  fuel_name  brl_price  year_of_reference month_of_reference  
0  Gasolina     2723.0               2021            January  
1  Gasolina     2484.0               2021            January  
2  Gasolina     2324.0               2021            January  
3  Gasolina     2199.0               2021            January  
4  Gasolina     2094.0               2021            January  
DEXBZUS head:
         date  exchange_rate
0  1995-01-01       0.846091
1  1995-02-01       0.841150
2  1995-03-01       0.890522
3  1995-04-01    

In [4]:
df_ipca['date'] = pd.to_datetime(df_ipca['date'])
df_ex['date'] = pd.to_datetime(df_ex['date'])

df_ipca.index = df_ipca['date']
df_ex.index = df_ex['date']

In [5]:
df_fipe = df_fipe.drop(columns=['Unnamed: 0'])

In [6]:
df_fipe['reference_date'] = pd.to_datetime(df_fipe['reference_date'], format='ISO8601')

df_fipe['sku'] = df_fipe.groupby(['brand_name', 'model_name', 'fuel_name', 'year']).ngroup()

In [7]:
# Retirar isso depois
df_fipe = df_fipe[df_fipe['reference_date'].dt.year > 2022]

In [8]:
df_fipe.head()

,reference_date,brand_name,model_name,year,fuel_name,brl_price,year_of_reference,month_of_reference,sku
125101,2023-09-01,Fiat,147 C/ CL,1987,Gasolina,4630.0,2023,September,2
125102,2023-09-01,Fiat,147 C/ CL,1986,Gasolina,4478.0,2023,September,1
125103,2023-09-01,Fiat,147 C/ CL,1985,Gasolina,3898.0,2023,September,0
125104,2023-09-01,Fiat,147 Furgão (todos),1987,Gasolina,2637.0,2023,September,5
125105,2023-09-01,Fiat,147 Furgão (todos),1986,Gasolina,2528.0,2023,September,4


In [9]:
df_fipe = df_fipe.drop(columns=['year_of_reference', 'month_of_reference'])

In [10]:
df_fipe.columns

Index(['reference_date', 'brand_name', 'model_name', 'year', 'fuel_name',
       'brl_price', 'sku'],
      dtype='str')

In [11]:
df_previsao_list = []

data_ref = pd.to_datetime('2026-04-01')

for sku in skus_teste:
    df = df_fipe.query("sku == @sku").copy()
    df = df.set_index('reference_date')

    meses_totais = relativedelta(data_ref, df.index.min()).years * 12 + relativedelta(data_ref, df.index.min()).months

    if meses_totais < 12:
        print(f"SKU: {sku} não tem dados suficientes ({meses_totais} meses apenas)")
        continue

    meses_esperados = pd.date_range(f'{df.index.min().year}-{df.index.min().month}', f'{data_ref.year}-{data_ref.month}', freq='MS')

    df = df.reindex(meses_esperados)

    df['brand_name'] = df['brand_name'].ffill()
    df['model_name'] = df['model_name'].ffill()
    df['year'] = df['year'].ffill()
    df['fuel_name'] = df['fuel_name'].ffill()

    df['brl_price'] = df['brl_price'].interpolate(method='linear')

    df = df.rename_axis('reference_date').reset_index()

    df.index = df['reference_date']
    df['sku'] = df['sku'].ffill()

    df_previsao_list.append(df)

df_previsao = pd.concat(df_previsao_list, ignore_index=True)

## Separar os dados em treino e teste para exógenas

In [12]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref + relativedelta(months=-tamanho_teste)

train_ipca = df_ipca[df_ipca.index <= start]

test_ipca = df_ipca[df_ipca.index > start]

train_ex = df_ex[df_ex.index <= start]

test_ex = df_ex[df_ex.index > start]

exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)


df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
df_prophet = df_prophet.rename(columns={
    'reference_date': 'ds',
    'brl_price': 'y'
})

df_train_prophet = df_prophet[
    df_prophet['ds'] <= start
]
df_test_prophet = df_prophet[
    df_prophet['ds'] > start
]

df_train_prophet = df_train_prophet.reset_index(drop=True)
df_test_prophet = df_test_prophet.reset_index(drop=True)

## Geradores de modelos SARIMAX e ETS

In [13]:
from model_generators import generate_ets_model, generate_sarimax_model, generate_prophet_model, feature_engineering, criar_ets_fipe_real, criar_sarimax_fipe_real, criar_prophet_fipe_real

## Modelos

#### Modelo do Câmbio

##### Prophet

In [14]:
df_train_exchange = df_train_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})
df_test_exchange = df_test_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})

model_exchange, info_exchange_prophet, best_value_exchange_prophet = generate_prophet_model(
    df_train_exchange,
    df_test_exchange,
    [],
    n_trials,
    metrica_erro,
    tolerancia_exog
)

model_exchange.fit(
    df_train_exchange,
)

forecast_exchange = model_exchange.predict(
    df_test_exchange[['ds']]
)

pd.concat([df_test_exchange['y'], forecast_exchange['yhat']], axis=1)


  0%|          | 0/3 [00:00<?, ?it/s]20:33:59 - cmdstanpy - INFO - Chain [1] start processing
20:33:59 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 0.77087:  33%|███▎      | 1/3 [00:00<00:00,  3.80it/s]20:33:59 - cmdstanpy - INFO - Chain [1] start processing
20:33:59 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 0.77087:  67%|██████▋   | 2/3 [00:00<00:00,  3.71it/s]20:33:59 - cmdstanpy - INFO - Chain [1] start processing
20:34:00 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 0.77087: 100%|██████████| 3/3 [00:00<00:00,  3.65it/s]
20:34:00 - cmdstanpy - INFO - Chain [1] start processing
20:34:00 - cmdstanpy - INFO - Chain [1] done processing


,y,yhat
0,5.341483,5.848810
1,5.455709,5.984547
2,5.331620,6.029607
3,5.198805,5.961596
4,5.229641,5.981215
5,5.033945,5.987325


##### SARIMAX

In [15]:
model, info_exchange_sarimax, best_value_exchange_sarimax = generate_sarimax_model(train_ex['exchange_rate'], test_ex['exchange_rate'], None, None, n_trials, metrica_erro, tolerancia_exog)

results = model.fit(disp=False)

forecasts_ex_sarimax = results.forecast(steps=len(test_ex['exchange_rate']))

  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 0.0917669:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will b

In [16]:
print(f"Valor da métrica: {best_value_exchange_sarimax}")
display(pd.concat([forecasts_ex_sarimax, test_ex['exchange_rate']], axis=1))

Valor da métrica: 0.09176694275567514


,predicted_mean,exchange_rate
2025-11-01,5.384488,5.341483
2025-12-01,5.384875,5.455709
2026-01-01,5.378925,5.331620
2026-02-01,5.369495,5.198805
2026-03-01,5.368779,5.229641
2026-04-01,5.369288,5.033945


#### Modelo do IPCA

##### Prophet

In [17]:
df_train_ipca = df_train_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})
df_test_ipca = df_test_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})

df_train_prophet = df_train_prophet.ffill()
df_test_prophet = df_test_prophet.ffill()

model_ipca, info_ipca_prophet, best_value_ipca_prophet = generate_prophet_model(
    df_train_ipca,
    df_test_ipca,
    [],
    n_trials,
    metrica_erro,
    tolerancia_exog
)

model_ipca.fit(
    df_train_ipca
)

forecast_ipca = model_ipca.predict(
    df_test_ipca[['ds']]
)

pd.concat([df_test_ipca['y'], forecast_ipca['yhat']], axis=1)

  0%|          | 0/3 [00:00<?, ?it/s]20:34:03 - cmdstanpy - INFO - Chain [1] start processing
20:34:03 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 0.254633:  33%|███▎      | 1/3 [00:00<00:00,  2.65it/s]20:34:03 - cmdstanpy - INFO - Chain [1] start processing
20:34:03 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 0.228878:  67%|██████▋   | 2/3 [00:00<00:00,  3.76it/s]20:34:03 - cmdstanpy - INFO - Chain [1] start processing
20:34:04 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 0.228878: 100%|██████████| 3/3 [00:00<00:00,  3.68it/s]
20:34:04 - cmdstanpy - INFO - Chain [1] start processing
20:34:04 - cmdstanpy - INFO - Chain [1] done processing


,y,yhat
0,0.18,0.324912
1,0.33,0.477466
2,0.33,0.333326
3,0.70,0.871716
4,0.88,0.423676
5,0.67,0.438645


##### SARIMAX

In [18]:
model, info_ipca_sarimax, best_value_ipca_sarimax = generate_sarimax_model(train_ipca['valor'], test_ipca['valor'], None, None, n_trials, metrica_erro, tolerancia_exog)

results = model.fit(disp=False)

forecasts_ipca_sarimax = results.forecast(steps=len(test_ipca['valor']))

  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 0.665489:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be

In [19]:
print(f"Valor da métrica: {best_value_ipca_sarimax}")
display(pd.concat([forecasts_ipca_sarimax, test_ipca['valor']], axis=1))

Valor da métrica: 0.3530001997534383


,predicted_mean,valor
2025-11-01,0.273818,0.18
2025-12-01,0.569074,0.33
2026-01-01,1.147177,0.33
2026-02-01,0.805506,0.70
2026-03-01,0.479050,0.88
2026-04-01,-0.597600,0.67


#### Selecionar melhor forecast das exógenas

In [20]:
metricas_ipca = np.array([best_value_ipca_prophet, best_value_ipca_sarimax])
metricas_exchange = np.array([best_value_exchange_prophet, best_value_exchange_sarimax])

idx_ipca = np.argmin(metricas_ipca)
idx_exchange = np.argmin(metricas_exchange)

forecast_ipca = pd.Series()
forecast_exchange = pd.Series()
modelo_escolhido_ipca = ''
modelo_escolhido_exchange = ''

if idx_ipca == 0:
    modelo_escolhido_ipca = 'Prophet'
    model_ipca = prophet.Prophet(**info_ipca_prophet)
    model_ipca.fit(pd.concat([df_train_ipca, df_test_ipca]))  
    future = model_ipca.make_future_dataframe(periods=horizonte_previsao, freq='MS')

    forecast_ipca = model_ipca.predict(
        future.tail(horizonte_previsao)
    )

    forecast_ipca.index = forecast_ipca['ds']
    forecast_ipca = forecast_ipca['yhat']
elif idx_ipca == 1:
    modelo_escolhido_ipca = 'SARIMAX'
    seasonal_order = (0, 0, 0, 0)

    if info_ipca_sarimax['seasonal']:
        seasonal_order = (
            info_ipca_sarimax['P'],
            info_ipca_sarimax['D'],
            info_ipca_sarimax['Q'],
            12
        )

    model_ipca = SARIMAX(
        pd.concat([train_ipca['valor'], test_ipca['valor']]),
        order=(
            info_ipca_sarimax['p'],
            info_ipca_sarimax['d'],
            info_ipca_sarimax['q']
        ),
        seasonal_order=seasonal_order,
        trend=info_ipca_sarimax['trend'],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results_ipca = model_ipca.fit(disp=False)

    forecast_ipca = results_ipca.forecast(steps=horizonte_previsao)

if idx_exchange == 0:
    modelo_escolhido_exchange = 'Prophet'
    model_exchange = prophet.Prophet(**info_exchange_prophet)
    model_exchange.fit(pd.concat([df_train_exchange, df_test_exchange]))  
    future = model_exchange.make_future_dataframe(periods=horizonte_previsao, freq='MS')

    forecast_exchange = model_exchange.predict(
        future.tail(horizonte_previsao)
    )

    forecast_exchange.index = forecast_exchange['ds']
    forecast_exchange = forecast_exchange['yhat']
elif idx_exchange == 1:
    modelo_escolhido_exchange = 'SARIMAX'
    seasonal_order = (0, 0, 0, 0)

    if info_exchange_sarimax['seasonal']:
        seasonal_order = (
            info_exchange_sarimax['P'],
            info_exchange_sarimax['D'],
            info_exchange_sarimax['Q'],
            12
        )

    model_exchange = SARIMAX(
        pd.concat([train_ex['exchange_rate'], test_ex['exchange_rate']]),
        order=(
            info_exchange_sarimax['p'],
            info_exchange_sarimax['d'],
            info_exchange_sarimax['q']
        ),
        seasonal_order=seasonal_order,
        trend=info_exchange_sarimax['trend'],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results_exchange = model_exchange.fit(disp=False)

    forecast_exchange = results_exchange.forecast(steps=horizonte_previsao)

forecast_exchange = forecast_exchange.rename("exchange_rate")
forecast_ipca = forecast_ipca.rename("valor")

exog_previsao = pd.concat([forecast_ipca, forecast_exchange], axis=1)

print(f"Modelo escolhido IPCA: {modelo_escolhido_ipca}")
print(f"Modelo escolhido taxa de câmbio: {modelo_escolhido_exchange}")

20:34:20 - cmdstanpy - INFO - Chain [1] start processing
20:34:20 - cmdstanpy - INFO - Chain [1] done processing


Modelo escolhido IPCA: Prophet
Modelo escolhido taxa de câmbio: SARIMAX


c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [21]:
display(forecast_exchange)

2026-05-01    4.989863
2026-06-01    4.980330
2026-07-01    4.966251
Freq: MS, Name: exchange_rate, dtype: float64

In [22]:
display(forecast_ipca)

ds
2026-05-01    0.322784
2026-06-01    0.164010
2026-07-01    0.285335
Name: valor, dtype: float64

#### Modelo da FIPE

##### Separar os dados em treino e teste para fipe

In [23]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref + relativedelta(months=-tamanho_teste)

train_ipca = df_ipca[df_ipca.index <= start]

test_ipca = df_ipca[df_ipca.index > start]

train_ex = df_ex[df_ex.index <= start]

test_ex = df_ex[df_ex.index > start]

exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)


df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
df_prophet = df_prophet.rename(columns={
    'reference_date': 'ds',
    'brl_price': 'y'
})

df_train_prophet = df_prophet[
    df_prophet['ds'] <= start
]
df_test_prophet = df_prophet[
    df_prophet['ds'] > start
]

df_train_prophet = df_train_prophet.reset_index(drop=True)
df_test_prophet = df_test_prophet.reset_index(drop=True)

In [25]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref - relativedelta(months=tamanho_teste)

previsoes_por_sku = {}
modelo_vencedor_por_sku = {}

for sku in skus_teste:
    df_sku_atual = df_previsao.query('sku == @sku')

    train = df_sku_atual[
        df_sku_atual['reference_date'] <= start
    ]
    test = df_sku_atual[
        df_sku_atual['reference_date'] > start
    ]

    train.index = train['reference_date']
    test.index = test['reference_date']

    train = train[train['reference_date'] >= f'{data_ref.year - 5}-01-01']

    modelo_ets, best_value_ets, forecast_ets = criar_ets_fipe_real(train, test, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    exog_train = exog_train[exog_train.index.isin(train['reference_date'])]

    modelo_sarimax, best_value_sarimax, forecast_sarimax = criar_sarimax_fipe_real(train, test, exog_train, exog_test, exog_previsao, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    df_sku_atual.index = df_sku_atual['reference_date']
    df_prophet = pd.concat([df_sku_atual, pd.concat([exog_train, exog_test])], axis=1, sort=False)
    df_prophet = df_prophet[['reference_date', 'brl_price', 'valor', 'exchange_rate']]
    df_prophet = df_prophet.rename(columns={
        'reference_date': 'ds',
        'brl_price': 'y'
    })

    df_train_prophet = df_prophet[
        df_prophet['ds'] <= start
    ]
    df_test_prophet = df_prophet[
        df_prophet['ds'] > start
    ]

    df_train_prophet = df_train_prophet.reset_index(drop=True)
    df_test_prophet = df_test_prophet.reset_index(drop=True)
    modelo_prophet, best_value_prophet, forecast_prophet = criar_prophet_fipe_real(df_train_prophet, df_test_prophet, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    resultados = {
        'ETS': {
            'erro': best_value_ets,
            'forecast': forecast_ets
        },
        'SARIMAX': {
            'erro': best_value_sarimax,
            'forecast': forecast_sarimax
        },
        'PROPHET': {
            'erro': best_value_prophet,
            'forecast': forecast_prophet
        }
    }

    melhor_modelo = min(
        resultados,
        key=lambda x: resultados[x]['erro']
    )

    previsoes_por_sku[sku] = resultados[melhor_modelo]['forecast']
    modelo_vencedor_por_sku[sku] = melhor_modelo

  0%|          | 0/3 [00:00<?, ?it/s]

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 70.1374:  33%|███▎      | 1/3 [00:00<00:00, 31.66it/s]
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


                 simulation  brl_price
reference_date                        
2025-11-01      4669.247218     4682.0
2025-12-01      4702.992576     4599.0
2026-01-01      4736.981817     4644.0
2026-02-01      4771.216703     4690.0
2026-03-01      4805.699010     4736.0
2026-04-01      4840.430525     4719.0
70.13740432182848
2026-05-01    4750.283429
2026-06-01    4780.907308
2026-07-01    4811.728611
Freq: MS, Name: simulation, dtype: float64


  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
Best trial: 0. Best value: 576.613:  33%|███▎      | 1/3 [00:00<00:00,  8.81it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
Best trial: 1. Best value: 52.1126:  67%|██████▋   | 2/3 [00:00<00:00,  7.60it/s]
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\v

                predicted_mean  brl_price
reference_date                           
2025-11-01         4689.423227     4682.0
2025-12-01         4686.584377     4599.0
2026-01-01         4711.632511     4644.0
2026-02-01         4745.993375     4690.0
2026-03-01         4766.124219     4736.0
2026-04-01         4832.144672     4719.0


...

                predicted_mean  brl_price
reference_date                           
2025-11-01         2501.098818     2511.0
2025-12-01         2486.147073     2507.0
2026-01-01         2477.847195     2502.0
2026-02-01         2503.332915     2496.0
2026-03-01         2640.321786     2490.0
2026-04-01         2676.154405     2481.0


c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


37.051413058985226
2026-05-01    2497.089253
2026-06-01    2458.388085
2026-07-01    2414.744088
Freq: MS, Name: predicted_mean, dtype: float64


  0%|          | 0/3 [00:00<?, ?it/s]20:37:50 - cmdstanpy - INFO - Chain [1] start processing
20:37:50 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 43.4816:  33%|███▎      | 1/3 [00:00<00:00,  4.88it/s]
20:37:50 - cmdstanpy - INFO - Chain [1] start processing
20:37:50 - cmdstanpy - INFO - Chain [1] done processing
20:37:50 - cmdstanpy - INFO - Chain [1] start processing
20:37:50 - cmdstanpy - INFO - Chain [1] done processing


43.48161286712624
        y         yhat
0  2511.0  2415.362594
1  2507.0  2409.129870
2  2502.0  2425.553806
3  2496.0  2423.319400
4  2490.0  2475.754313
5  2481.0  2475.925536
ds
2026-05-01    2496.831513
2026-06-01    2491.927410
2026-07-01    2474.164218
Name: yhat, dtype: float64


In [26]:
previsoes_por_sku

{0: 2026-05-01    4762.482551
 2026-06-01    4828.531364
 2026-07-01    4855.527523
 Freq: MS, Name: predicted_mean, dtype: float64,
 1: 2026-05-01    5779.226207
 2026-06-01    5693.319022
 2026-07-01    5734.589262
 Freq: MS, Name: predicted_mean, dtype: float64,
 2: 2026-05-01    6016.926346
 2026-06-01    6078.174240
 2026-07-01    6035.709069
 Freq: MS, Name: simulation, dtype: float64,
 3: 2026-05-01    2167.195043
 2026-06-01    2165.378168
 2026-07-01    2156.899770
 Freq: MS, Name: simulation, dtype: float64,
 4: 2026-05-01    2479.217582
 2026-06-01    2473.932034
 2026-07-01    2459.740862
 Freq: MS, Name: simulation, dtype: float64}

In [27]:
modelo_vencedor_por_sku

{0: 'SARIMAX', 1: 'SARIMAX', 2: 'ETS', 3: 'ETS', 4: 'ETS'}